# Colab fallback entry point

This notebook is the **fallback execution path** for this project, documented in
`environment.md`. Everything here runs locally by default (RTX 5060, 8 GB VRAM, `ViT-B/16`,
`-b 1`) via `scripts\run_all.ps1` / `scripts/run_one.sh`. Use this notebook instead when:

- a run does not fit in 8 GB locally (e.g. a larger backbone, or a machine without a
  CUDA 12.8+/Blackwell-capable GPU), or
- local CUDA is unavailable or broken and a GPU is still needed to reproduce a result.

It clones this repository, installs the pinned dependencies, mounts Google Drive for the
dataset root (see `docs/DATASETS.md` for how the five datasets are laid out), runs **one**
dataset/method combination through the same `scripts/run_one.sh` entry point the local
PowerShell scripts use (so results are directly comparable — same code path, same JSON
record format), and regenerates the aggregate table and figures from whatever records exist
under `results/raw/`.

This is a single-run cell, not the full 15-run sweep — rerun cell 3 with different
`--dataset` / `--method` / `--config` values for each combination you need, the same way
`scripts/run_all.ps1` loops over them locally.

In [ ]:
# Setup: clone the repo, install pinned deps, mount Drive for the dataset root.
#
# The repository is PRIVATE, so an unauthenticated clone fails here. Either clone with a
# personal access token -- https://<token>@github.com/macwave12/... -- or copy the repo into
# Drive and skip this line. Making it public is not an option: the repo vendors upstream
# tta-vlm, which publishes no license (see NOTICE).
!git clone https://github.com/macwave12/deep_project_calibration_aware_tta.git
%cd deep_project_calibration_aware_tta
!pip install -q -r requirements.txt
from google.colab import drive; drive.mount('/content/drive')
DATA_ROOT = '/content/drive/MyDrive/deep_project_data'
# Expected layout under DATA_ROOT: dtd/, oxford_flowers/, oxford_pets/, eurosat/,
# fgvc_aircraft/ — see docs/DATASETS.md for the exact subfolder contents and sources.
#
# This must be a real OS environment variable, not just a Python variable: `run_one.sh`'s
# `--data-root` flag only feeds upstream's `--data` CLI argument, but
# `data/fewshot_datasets.py` independently reads `os.environ["DATA_ROOT"]` for the
# split-JSON paths and silently falls back to a local-machine-specific default path if it
# is unset. `%env` (unlike a plain Python assignment) exports into the OS environment that
# the `!`-prefixed shell commands below inherit.
%env DATA_ROOT=$DATA_ROOT

In [ ]:
# Single run: one dataset x one method, through the same entry point run_all.ps1 uses
# locally (run_one.sh maps these onto upstream's --data / --test_sets / --algorithm flags,
# and always passes -b 1 -j 0, matching the local runs exactly). --data-root sets
# upstream's --data flag; the DATA_ROOT environment variable data/fewshot_datasets.py
# reads independently was already exported by %env in the setup cell above.
!bash scripts/run_one.sh --dataset eurosat --method cal_tda \
    --config configs/method/cal_tda.yaml --data-root $DATA_ROOT
#
# For the baselines on the same dataset, rerun with:
#   --method clipzs   (no --config)
#   --method tda --config configs/method/tda.yaml
# Swap --dataset for any of: dtd, flower102, pets, eurosat, aircraft.

In [ ]:
# Inspect: rebuild the summary table and figures from whatever run records exist under
# results/raw/ (local runs synced in, or runs produced by cell 3 above), then preview the
# headline accuracy-vs-ECE figure inline.
!python analysis/aggregate.py && python analysis/make_figures.py
from IPython.display import Image
Image('analysis/figures/accuracy_vs_ece.png')